# Bronze Layer Ingestion Notebook
## Purpose: Process raw data in Bronze layer

**Note: Uses Spark tables (DBFS disabled in Free Edition)**

In [ ]:
# Bronze Layer Processing
from pyspark.sql.functions import col, upper, trim, row_number, current_timestamp, lit
from pyspark.sql.window import Window
from datetime import datetime

# spark session already exists in Databricks notebooks

BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Starting Bronze layer processing - Batch ID: {BATCH_ID}")

In [ ]:
# Process Orders
print("\n--- Processing Orders ---")

orders = spark.table("bronze_orders")

# Deduplicate
w = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())
orders = orders.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

# Clean
orders = (orders
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("status", trim(col("status")))
    .withColumn("payment_mode", upper(trim(col("payment_mode")))))

orders.write.mode("overwrite").saveAsTable("silver_orders_clean")
print(f"✓ silver_orders_clean: {orders.count()} rows")

In [ ]:
# Process Order Items
print("\n--- Processing Order Items ---")

items = spark.table("bronze_order_items")

# Aggregate per order
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round
items_agg = items.groupBy("order_id").agg(
    count("item_id").alias("item_count"),
    spark_round(spark_sum(col("quantity") * col("item_price")), 2).alias("items_subtotal")
)

items_agg.write.mode("overwrite").saveAsTable("silver_order_items_agg")
print(f"✓ silver_order_items_agg: {items_agg.count()} rows")

In [ ]:
# Process Refunds
print("\n--- Processing Refunds ---")

refunds = spark.table("bronze_refunds")

# Deduplicate
w = Window.partitionBy("refund_id").orderBy(col("_ingested_at").desc())
refunds = refunds.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

# Aggregate
refunds_agg = refunds.groupBy("order_id").agg(
    lit(True).alias("has_refund"),
    spark_round(spark_sum("refund_amount"), 2).alias("refund_amount")
)

refunds_agg.write.mode("overwrite").saveAsTable("silver_refunds_agg")
print(f"✓ silver_refunds_agg: {refunds_agg.count()} rows")

In [ ]:
# Process Delivery Events
print("\n--- Processing Delivery Events ---")

events = spark.table("bronze_delivery_events")
riders = spark.table("bronze_riders")

# Deduplicate events
w = Window.partitionBy("order_id", "event_type").orderBy(col("_ingested_at").desc())
events = events.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

# Join with riders
events_with_riders = events.join(
    riders.select("rider_id", col("city").alias("rider_city"), "shift_type"),
    on="rider_id", how="left"
)

events_with_riders.write.mode("overwrite").saveAsTable("silver_delivery_events")
print(f"✓ silver_delivery_events: {events_with_riders.count()} rows")

In [ ]:
# Process Restaurants
print("\n--- Processing Restaurants ---")

restaurants = spark.table("bronze_restaurants")

# Deduplicate
w = Window.partitionBy("restaurant_id").orderBy(col("_ingested_at").desc())
restaurants = restaurants.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

restaurants.write.mode("overwrite").saveAsTable("silver_restaurants")
print(f"✓ silver_restaurants: {restaurants.count()} rows")

In [ ]:
# Process Support Tickets
print("\n--- Processing Support Tickets ---")

tickets = spark.table("bronze_support_tickets")
orders = spark.table("silver_orders_clean")

# Deduplicate
w = Window.partitionBy("ticket_id").orderBy(col("_ingested_at").desc())
tickets = tickets.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

# Join with orders to get restaurant_id
tickets_with_orders = tickets.join(
    orders.select("order_id", "restaurant_id", "city"),
    on="order_id", how="left"
)

tickets_with_orders.write.mode("overwrite").saveAsTable("silver_support_tickets")
print(f"✓ silver_support_tickets: {tickets_with_orders.count()} rows")

In [ ]:
# Summary
print("\n" + "="*60)
print("BRONZE LAYER COMPLETE")
print("="*60)
print("\nSilver tables created:")
print("  - silver_orders_clean")
print("  - silver_order_items_agg")
print("  - silver_refunds_agg")
print("  - silver_delivery_events")
print("  - silver_restaurants")
print("  - silver_support_tickets")

spark.stop()